# **IRS Form Collection Pipeline**
This notebook collects publicly available IRS tax forms and instructions.

## Mount Google Drive and Initialize Project Directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

project_root = Path("/content/drive/MyDrive/newstart_ai")

for folder in [
    "data/raw/uscis",
    "data/raw/dmv",
    "data/raw/irs",
    "data/metadata"
]:
    path = project_root / folder
    path.mkdir(parents=True, exist_ok=True)
    print(path, path.exists())

/content/drive/MyDrive/newstart_ai/data/raw/uscis True
/content/drive/MyDrive/newstart_ai/data/raw/dmv True
/content/drive/MyDrive/newstart_ai/data/raw/irs True
/content/drive/MyDrive/newstart_ai/data/metadata True


## Verify Project structure
Before downloading any files, I verified the expected project directories already existed and were properly populated.

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("Drive exists:", drive.exists())
print("Top-level Drive folders:")
for item in list(drive.iterdir())[:10]:
    print("-", item.name)

Drive exists: True
Top-level Drive folders:
- AAI 5001.gslides
- Copy of AAI-510 M5 Lab Walkthrough.ipynb
- Final Project Section 1 - Team 1.gslides
- Untitled0.ipynb
- Team01_FinalReport.gdoc
- MSAAI 521 Final Presentation.gslides
- Team01_Presentation-Video - Made with Clipchamp_1764794122392.mp4
- IoTAgricultureProject
- MSAAI530A62.pdf
- MSAAI530A6.html


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/newstart_ai")

print("Project exists:", project.exists())

for item in project.iterdir():
    print(item.name)

Project exists: True
data


In [ ]:
from pathlib import Path

raw = Path("/content/drive/MyDrive/newstart_ai/data/raw")

for agency in ["uscis", "dmv", "ssa", "irs"]:

    folder = raw / agency

    print("\nAgency:", agency)
    print("Exists:", folder.exists())

    if folder.exists():
        print("PDF count:", len(list(folder.glob("*.pdf"))))


Agency: uscis
Exists: True
PDF count: 256

Agency: dmv
Exists: True
PDF count: 277

Agency: ssa
Exists: True
PDF count: 0

Agency: irs
Exists: True
PDF count: 0


In [ ]:
metadata = Path("/content/drive/MyDrive/newstart_ai/data/metadata")

print("Metadata exists:", metadata.exists())

if metadata.exists():
    for file in metadata.iterdir():
        print(file.name)

Metadata exists: True
dmv_metadata.csv
uscis_metadata.csv


## Import Required Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin, urlparse
import pandas as pd
import re
import time

In [ ]:
irs_folder = "/content/drive/MyDrive/newstart_ai/data/raw/irs"

irs_folder.mkdir(parents=True, exist_ok=True)

print("IRS folder:")
print(irs_folder)

IRS folder:
/content/drive/MyDrive/newstart_ai/data/raw/irs


## Explore the IRS Forms Website
Before building the crawler, I inspected the IRS website to see how the forms were structured and organized

In [ ]:
url = "https://www.irs.gov/forms-instructions"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<html lang="en" dir="ltr" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# ">
  <head>
    <meta charset="utf-8" />
<meta name="description" content="Access IRS f


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    links.append(a["href"])


print("Total links:", len(links))

for link in links[:20]:
    print(link)

Total links: 240
#main-content
/
/es/forms-instructions
/zh-hans/forms-instructions
/zh-hant/forms-instructions
/ko/forms-instructions
/ru/forms-instructions
/vi/forms-instructions
/ht/forms-instructions
/help/let-us-help-you
/newsroom
/tax-professionals
https://sa.www4.irs.gov/ola
https://sa.www4.irs.gov/bola
https://sa.www4.irs.gov/taxpro
/your-account
https://sa.www4.irs.gov/ola
https://sa.www4.irs.gov/bola
https://sa.www4.irs.gov/taxpro
/your-account


## Identify Candidate Form pages
Extract links that looked like they referenced IRS forms and publications

In [ ]:
base_url = "https://www.irs.gov"

form_links = []

for link in links:

    full_url = urljoin(base_url, link)

    if "/forms-pubs/" in full_url:
        form_links.append(full_url)


form_links = list(set(form_links))

print("Form links found:", len(form_links))

for link in form_links[:20]:
    print(link)

Form links found: 21
https://www.irs.gov/forms-pubs/about-form-w-2
https://www.irs.gov/forms-pubs/about-form-9465
https://www.irs.gov/forms-pubs/prior-year
https://www.irs.gov/forms-pubs/about-form-1040
https://www.irs.gov/forms-pubs/about-form-4547
https://www.irs.gov/forms-pubs/about-form-1040x
https://www.irs.gov/forms-pubs/more-information
https://www.irs.gov/forms-pubs/about-form-941
https://www.irs.gov/forms-pubs/schedules-for-form-1040
https://www.irs.gov/forms-pubs/about-form-4506-t
https://www.irs.gov/forms-pubs/browser-friendly
https://www.irs.gov/forms-pubs/accessible-irs-tax-products
https://www.irs.gov/forms-pubs/about-form-w-7
https://www.irs.gov/forms-pubs/about-form-w-4
https://www.irs.gov/forms-pubs/about-form-w-9
https://www.irs.gov/forms-pubs/changes-to-current-forms-publications
https://www.irs.gov/forms-pubs/about-form-2848
https://www.irs.gov/forms-pubs/order-products
https://www.irs.gov/forms-pubs/mobile-friendly-forms
https://www.irs.gov/forms-pubs/ebook


In [ ]:
for link in links:
    if "form" in link.lower():
        print(link)

/es/forms-instructions
/zh-hans/forms-instructions
/zh-hant/forms-instructions
/ko/forms-instructions
/ru/forms-instructions
/vi/forms-instructions
/ht/forms-instructions
/filing/individuals/update-my-information
/forms-instructions
/forms-instructions
/forms-pubs/about-form-1040
/forms-pubs/about-form-w-9
/forms-pubs/about-form-4506-t
/forms-pubs/about-form-w-4
/forms-pubs/about-form-941
/forms-pubs/about-form-w-2
/forms-pubs/about-form-9465
/forms-pubs/about-form-1040x
/forms-pubs/about-form-2848
/forms-pubs/about-form-w-7
/filing/individuals/update-my-information
/forms-instructions
/forms-pubs/about-form-1040
/forms-pubs/about-form-w-9
/forms-pubs/about-form-4506-t
/forms-pubs/about-form-w-4
/forms-pubs/about-form-941
/forms-pubs/about-form-w-2
/forms-pubs/about-form-9465
/forms-pubs/about-form-1040x
/forms-pubs/about-form-2848
/forms-pubs/about-form-w-7
/forms-instructions
/forms-instructions
/forms-pubs/prior-year
/forms-pubs/accessible-irs-tax-products
/forms-pubs/ebook
/forms-p

In [ ]:
irs_forms_url = "https://www.irs.gov/forms-instructions"

response = requests.get(irs_forms_url)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

for a in soup.find_all("a", href=True):
    text = a.get_text(strip=True)

    if "form" in text.lower():
        print(text, " --> ", a["href"])

200
Update your information  -->  /filing/individuals/update-my-information
Forms  -->  /forms-instructions
Form 1040  -->  /forms-pubs/about-form-1040
Form 1040 Instructions  -->  /instructions/i1040gi
Form W-9  -->  /forms-pubs/about-form-w-9
Form 4506-T  -->  /forms-pubs/about-form-4506-t
Form W-4  -->  /forms-pubs/about-form-w-4
Form 941  -->  /forms-pubs/about-form-941
Form W-2  -->  /forms-pubs/about-form-w-2
Form 9465  -->  /forms-pubs/about-form-9465
Form 1040-X  -->  /forms-pubs/about-form-1040x
Form 2848  -->  /forms-pubs/about-form-2848
Form W-7  -->  /forms-pubs/about-form-w-7
Update your information  -->  /filing/individuals/update-my-information
Forms  -->  #
Form 1040  -->  /forms-pubs/about-form-1040
Form 1040 Instructions  -->  /instructions/i1040gi
Form W-9  -->  /forms-pubs/about-form-w-9
Form 4506-T  -->  /forms-pubs/about-form-4506-t
Form W-4  -->  /forms-pubs/about-form-w-4
Form 941  -->  /forms-pubs/about-form-941
Form W-2  -->  /forms-pubs/about-form-w-2
Form 94

## Explore the Forms and Publications
Ths looks at the centralized index of links to collect the available PDF links

In [ ]:
irs_index = "https://www.irs.gov/forms-instructions-and-publications"

response = requests.get(irs_index)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")


irs_pdf_links = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    if "/pub/irs-pdf/" in href and href.endswith(".pdf"):

        full_url = urljoin("https://www.irs.gov", href)
        irs_pdf_links.append(full_url)


# remove duplicates
irs_pdf_links = list(set(irs_pdf_links))


print("PDFs found:", len(irs_pdf_links))

for pdf in irs_pdf_links[:20]:
    print(pdf)

200
PDFs found: 25
https://www.irs.gov/pub/irs-pdf/p3.pdf
https://www.irs.gov/pub/irs-pdf/p1ar.pdf
https://www.irs.gov/pub/irs-pdf/p1.pdf
https://www.irs.gov/pub/irs-pdf/p1zhs.pdf
https://www.irs.gov/pub/irs-pdf/p1zht.pdf
https://www.irs.gov/pub/irs-pdf/p5.pdf
https://www.irs.gov/pub/irs-pdf/p1ru.pdf
https://www.irs.gov/pub/irs-pdf/p1fr.pdf
https://www.irs.gov/pub/irs-pdf/p1fa.pdf
https://www.irs.gov/pub/irs-pdf/p5sp.pdf
https://www.irs.gov/pub/irs-pdf/p1ja.pdf
https://www.irs.gov/pub/irs-pdf/p1ur.pdf
https://www.irs.gov/pub/irs-pdf/p1guj.pdf
https://www.irs.gov/pub/irs-pdf/p1it.pdf
https://www.irs.gov/pub/irs-pdf/f11c.pdf
https://www.irs.gov/pub/irs-pdf/p1pa.pdf
https://www.irs.gov/pub/irs-pdf/p1tl.pdf
https://www.irs.gov/pub/irs-pdf/p1vie.pdf
https://www.irs.gov/pub/irs-pdf/p1ht.pdf
https://www.irs.gov/pub/irs-pdf/p1pt.pdf


## Discover Individual IRS Form Pages
Find the right pages and prepare them for crawling

In [ ]:
# Find IRS form detail pages

irs_form_pages = []

for link in links:

    full_url = urljoin(
        "https://www.irs.gov",
        link
    )

    if "/forms-pubs/about-form-" in full_url:
        irs_form_pages.append(full_url)


irs_form_pages = list(set(irs_form_pages))


print("IRS form pages found:", len(irs_form_pages))

for page in irs_form_pages:
    print(page)

IRS form pages found: 11
https://www.irs.gov/forms-pubs/about-form-w-2
https://www.irs.gov/forms-pubs/about-form-4506-t
https://www.irs.gov/forms-pubs/about-form-9465
https://www.irs.gov/forms-pubs/about-form-w-7
https://www.irs.gov/forms-pubs/about-form-2848
https://www.irs.gov/forms-pubs/about-form-w-4
https://www.irs.gov/forms-pubs/about-form-1040
https://www.irs.gov/forms-pubs/about-form-w-9
https://www.irs.gov/forms-pubs/about-form-4547
https://www.irs.gov/forms-pubs/about-form-1040x
https://www.irs.gov/forms-pubs/about-form-941


## Collect Downloadable PDF Documents

In [ ]:
irs_pdf_links = []

for page in irs_form_pages:

    print("Checking:", page)

    response = requests.get(page)

    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/pub/irs-pdf/" in href and href.endswith(".pdf"):

            full_url = urljoin(
                "https://www.irs.gov",
                href
            )

            irs_pdf_links.append(full_url)


# Remove duplicates
irs_pdf_links = list(set(irs_pdf_links))


print("\nTotal IRS PDFs found:", len(irs_pdf_links))


for pdf in irs_pdf_links:
    print(pdf)

Checking: https://www.irs.gov/forms-pubs/about-form-w-2
Checking: https://www.irs.gov/forms-pubs/about-form-4506-t
Checking: https://www.irs.gov/forms-pubs/about-form-9465
Checking: https://www.irs.gov/forms-pubs/about-form-w-7
Checking: https://www.irs.gov/forms-pubs/about-form-2848
Checking: https://www.irs.gov/forms-pubs/about-form-w-4
Checking: https://www.irs.gov/forms-pubs/about-form-1040
Checking: https://www.irs.gov/forms-pubs/about-form-w-9
Checking: https://www.irs.gov/forms-pubs/about-form-4547
Checking: https://www.irs.gov/forms-pubs/about-form-1040x
Checking: https://www.irs.gov/forms-pubs/about-form-941

Total IRS PDFs found: 40
https://www.irs.gov/pub/irs-pdf/f1040x.pdf
https://www.irs.gov/pub/irs-pdf/fw7.pdf
https://www.irs.gov/pub/irs-pdf/i1040x.pdf
https://www.irs.gov/pub/irs-pdf/f941sb.pdf
https://www.irs.gov/pub/irs-pdf/f4506t.pdf
https://www.irs.gov/pub/irs-pdf/f1040s1a.pdf
https://www.irs.gov/pub/irs-pdf/i941.pdf
https://www.irs.gov/pub/irs-pdf/f1040s3.pdf
https:/

## Filter the Dataset
Not every downloadable document met the requirements for the dataset, so it removes schedules, and other documents that werent primary forms or instructions.

In [ ]:
filtered_irs_pdf_links = []

for url in irs_pdf_links:

    filename = url.split("/")[-1].lower()

    # Keep forms and instructions
    if filename.startswith("f") or filename.startswith("i"):
        filtered_irs_pdf_links.append(url)


filtered_irs_pdf_links = list(set(filtered_irs_pdf_links))


print("Before:", len(irs_pdf_links))
print("After:", len(filtered_irs_pdf_links))

for pdf in filtered_irs_pdf_links:
    print(pdf)

Before: 40
After: 34
https://www.irs.gov/pub/irs-pdf/f1040x.pdf
https://www.irs.gov/pub/irs-pdf/fw7.pdf
https://www.irs.gov/pub/irs-pdf/i1040x.pdf
https://www.irs.gov/pub/irs-pdf/f941sb.pdf
https://www.irs.gov/pub/irs-pdf/f4506t.pdf
https://www.irs.gov/pub/irs-pdf/f1040s1a.pdf
https://www.irs.gov/pub/irs-pdf/i941.pdf
https://www.irs.gov/pub/irs-pdf/f1040s3.pdf
https://www.irs.gov/pub/irs-pdf/iw2w3.pdf
https://www.irs.gov/pub/irs-pdf/fw4.pdf
https://www.irs.gov/pub/irs-pdf/f1040s1.pdf
https://www.irs.gov/pub/irs-pdf/f941sr.pdf
https://www.irs.gov/pub/irs-pdf/i1040gi.pdf
https://www.irs.gov/pub/irs-pdf/f4547.pdf
https://www.irs.gov/pub/irs-pdf/i4547.pdf
https://www.irs.gov/pub/irs-pdf/f2159.pdf
https://www.irs.gov/pub/irs-pdf/iw7.pdf
https://www.irs.gov/pub/irs-pdf/f1040.pdf
https://www.irs.gov/pub/irs-pdf/f941sd.pdf
https://www.irs.gov/pub/irs-pdf/fw2.pdf
https://www.irs.gov/pub/irs-pdf/f1040s2.pdf
https://www.irs.gov/pub/irs-pdf/f8508.pdf
https://www.irs.gov/pub/irs-pdf/i941sd.pdf
http

In [ ]:
final_irs_pdf_links = []

remove_keywords = [
    "s1",
    "s1a",
    "s2",
    "s3",
    "f1040s.pdf",
    "941sb",
    "941sr",
    "941sd",
    "941sb",
    "941sr",
    "941sd"
]


for url in filtered_irs_pdf_links:

    filename = url.split("/")[-1].lower()

    remove = False

    for keyword in remove_keywords:
        if keyword in filename:
            remove = True

    if not remove:
        final_irs_pdf_links.append(url)


print("Final IRS PDFs:", len(final_irs_pdf_links))

for pdf in final_irs_pdf_links:
    print(pdf)

Final IRS PDFs: 23
https://www.irs.gov/pub/irs-pdf/f1040x.pdf
https://www.irs.gov/pub/irs-pdf/fw7.pdf
https://www.irs.gov/pub/irs-pdf/i1040x.pdf
https://www.irs.gov/pub/irs-pdf/f4506t.pdf
https://www.irs.gov/pub/irs-pdf/i941.pdf
https://www.irs.gov/pub/irs-pdf/iw2w3.pdf
https://www.irs.gov/pub/irs-pdf/fw4.pdf
https://www.irs.gov/pub/irs-pdf/i1040gi.pdf
https://www.irs.gov/pub/irs-pdf/f4547.pdf
https://www.irs.gov/pub/irs-pdf/i4547.pdf
https://www.irs.gov/pub/irs-pdf/f2159.pdf
https://www.irs.gov/pub/irs-pdf/iw7.pdf
https://www.irs.gov/pub/irs-pdf/f1040.pdf
https://www.irs.gov/pub/irs-pdf/fw2.pdf
https://www.irs.gov/pub/irs-pdf/f8508.pdf
https://www.irs.gov/pub/irs-pdf/f941.pdf
https://www.irs.gov/pub/irs-pdf/fw9.pdf
https://www.irs.gov/pub/irs-pdf/i2848.pdf
https://www.irs.gov/pub/irs-pdf/f13844.pdf
https://www.irs.gov/pub/irs-pdf/i9465.pdf
https://www.irs.gov/pub/irs-pdf/f2848.pdf
https://www.irs.gov/pub/irs-pdf/iw9.pdf
https://www.irs.gov/pub/irs-pdf/f9465.pdf


## Download IRS Documents

In [ ]:
import requests
from pathlib import Path
import time


irs_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/irs")

irs_folder.mkdir(parents=True, exist_ok=True)


downloaded = 0
failed = []


for url in final_irs_pdf_links:

    filename = url.split("/")[-1]

    filepath = irs_folder / filename

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:

            with open(filepath, "wb") as f:
                f.write(response.content)

            downloaded += 1

        else:
            failed.append((url, response.status_code))


    except Exception as e:
        failed.append((url, str(e)))


    time.sleep(0.5)


print("Download complete")
print("Downloaded:", downloaded)
print("Failed:", len(failed))

if failed:
    print("\nFailures:")
    for item in failed:
        print(item)

Download complete
Downloaded: 23
Failed: 0


In [ ]:
from pathlib import Path

irs_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/irs")

pdfs = list(irs_folder.glob("*.pdf"))

print("IRS PDFs:", len(pdfs))

for pdf in pdfs[:10]:
    print(pdf)

IRS PDFs: 23
/content/drive/MyDrive/newstart_ai/data/raw/irs/f1040x.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/fw7.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/i1040x.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/f4506t.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/i941.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/iw2w3.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/fw4.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/i1040gi.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/f4547.pdf
/content/drive/MyDrive/newstart_ai/data/raw/irs/i4547.pdf


## Generate Document Metadata

In [ ]:
import pandas as pd
import re
from pathlib import Path


irs_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/irs")
metadata_folder = Path("/content/drive/MyDrive/newstart_ai/data/metadata")

metadata_folder.mkdir(parents=True, exist_ok=True)


def extract_form_number(filename):
    """
    Extract IRS form identifier.

    Examples:
    f1040.pdf -> 1040
    fw4.pdf -> W-4
    f4506t.pdf -> 4506-T
    """

    name = filename.lower().replace(".pdf", "")

    if name.startswith("f"):

        form = name[1:]

        # Handle W forms
        if form.startswith("w"):
            return form.upper().replace("W", "W-")

        return form.upper()

    elif name.startswith("i"):

        form = name[1:]

        if form.startswith("w"):
            return form.upper().replace("W", "W-")

        return form.upper()

    return None



def classify_document_type(filename):

    name = filename.lower()

    if name.startswith("i"):
        return "instructions"

    return "form"



irs_metadata = []


for pdf in irs_folder.glob("*.pdf"):

    irs_metadata.append({

        "filename": pdf.name,
        "agency": "IRS",
        "form_number": extract_form_number(pdf.name),
        "document_type": classify_document_type(pdf.name),
        "url": f"https://www.irs.gov/pub/irs-pdf/{pdf.name}"

    })


irs_df = pd.DataFrame(irs_metadata)


output_file = metadata_folder / "irs_metadata.csv"

irs_df.to_csv(output_file, index=False)


print("IRS metadata saved!")
print("Rows:", len(irs_df))

irs_df.head(10)

IRS metadata saved!
Rows: 23


,filename,agency,form_number,document_type,url
0,f1040x.pdf,IRS,1040X,form,https://www.irs.gov/pub/irs-pdf/f1040x.pdf
1,fw7.pdf,IRS,W-7,form,https://www.irs.gov/pub/irs-pdf/fw7.pdf
2,i1040x.pdf,IRS,1040X,instructions,https://www.irs.gov/pub/irs-pdf/i1040x.pdf
3,f4506t.pdf,IRS,4506T,form,https://www.irs.gov/pub/irs-pdf/f4506t.pdf
4,i941.pdf,IRS,941,instructions,https://www.irs.gov/pub/irs-pdf/i941.pdf
5,iw2w3.pdf,IRS,W-2W-3,instructions,https://www.irs.gov/pub/irs-pdf/iw2w3.pdf
6,fw4.pdf,IRS,W-4,form,https://www.irs.gov/pub/irs-pdf/fw4.pdf
7,i1040gi.pdf,IRS,1040GI,instructions,https://www.irs.gov/pub/irs-pdf/i1040gi.pdf
8,f4547.pdf,IRS,4547,form,https://www.irs.gov/pub/irs-pdf/f4547.pdf
9,i4547.pdf,IRS,4547,instructions,https://www.irs.gov/pub/irs-pdf/i4547.pdf
